In [1]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image

# ======================
# 1) تابع محاسبه DICE
# ======================
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)      # تبدیل لاجیت خروجی به محدوده [0,1]
    pred = (pred > 0.5).float()       # باینری کردن خروجی
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()

# ======================
# 2) تنظیم seed
# ======================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================
# 3) تعریف دیتاست
# ======================
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")  # اگر تصاویر رنگی‌اند
        mask = Image.open(mask_path).convert("L")    # ماسک معمولاً خاکستری یا باینری

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # باینری کردن ماسک (اگر داده‌های شما 0 و 255 هستند)
        mask = (mask > 0.5).float()

        return image, mask

# ======================
# 4) آدرس فولدر تصاویر
# ======================
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها برابر نیست."

# ======================
# 5) تقسیم داده‌ها: 80% Train, 20% Test
# ======================
total_size = len(images_list)
train_size = int(total_size * 0.8)
test_size  = total_size - train_size

train_images = images_list[:train_size]
train_masks  = masks_list[:train_size]
test_images  = images_list[train_size:]
test_masks   = masks_list[train_size:]

# ======================
# 6) تعریف ترنسفورم‌ها
# ======================
# ترنسفورم برای داده‌های تست
transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم پایه برای داده‌های آموزش
transform_train_base = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم آگومنت‌شده برای داده‌های آموزش
transform_train_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor()
])

# ======================
# 7) ساخت دیتاست
# ======================
# دیتاست اصلی
train_dataset_base = CorneaDataset(train_images, train_masks, transform=transform_train_base)
# دیتاست آگومنت‌شده
train_dataset_aug  = CorneaDataset(train_images, train_masks, transform=transform_train_aug)
# ترکیب دیتاست اصلی و آگومنت شده
train_dataset      = ConcatDataset([train_dataset_base, train_dataset_aug])

test_dataset       = CorneaDataset(test_images, test_masks, transform=transform_test)




import torch
import torch.nn as nn
import torch.optim as optim

# ======================
# 9) تعریف مدل AutoEncoder ساده
# ======================
class SimpleAutoEncoder(nn.Module):
    def __init__(self):
        super(SimpleAutoEncoder, self).__init__()

        # بخش Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),   # 256 -> 128

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),   # 128 -> 64

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),   # 64 -> 32
        )

        # بخش Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # 32 -> 64
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),  # 64 -> 128
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2),   # 128 -> 256
            nn.ReLU(inplace=True),

            nn.Conv2d(8, 1, kernel_size=1),  # خروجی یک کاناله (ماسک)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# ======================
# 10) تابع آموزش و اعتبارسنجی
# ======================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        # محاسبه DICE
        dice_score = dice_coefficient(outputs, masks)

        epoch_loss += loss.item()
        epoch_dice += dice_score.item()

    return epoch_loss / len(dataloader), epoch_dice / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks  = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)
            dice_score = dice_coefficient(outputs, masks)

            val_loss += loss.item()
            val_dice += dice_score.item()

    return val_loss / len(dataloader), val_dice / len(dataloader)


In [2]:
# ======================
# 8) تعریف دیتالودر
# ======================
batch_size = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 11) تعریف دستگاه، ساخت مدل، معیار و بهینه‌ساز
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleAutoEncoder().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) حلقه آموزش
# ======================
num_epochs = 20  # به دلخواه تغییر دهید

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 13) ذخیره مدل (در صورت نیاز)
# ======================
torch.save(model.state_dict(), "autoencoder_cornea.pth")


In [2]:
# ======================
# 8) تعریف دیتالودر
# ======================
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 11) تعریف دستگاه، ساخت مدل، معیار و بهینه‌ساز
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleAutoEncoder().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) حلقه آموزش
# ======================
num_epochs = 20  # به دلخواه تغییر دهید

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 13) ذخیره مدل (در صورت نیاز)
# ======================
torch.save(model.state_dict(), "autoencoder_cornea.pth")


In [3]:
# ======================
# 8) تعریف دیتالودر
# ======================
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 11) تعریف دستگاه، ساخت مدل، معیار و بهینه‌ساز
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleAutoEncoder().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) حلقه آموزش
# ======================
num_epochs = 20  # به دلخواه تغییر دهید

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 13) ذخیره مدل (در صورت نیاز)
# ======================
torch.save(model.state_dict(), "autoencoder_cornea.pth")


In [4]:
# ======================
# 8) تعریف دیتالودر
# ======================
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 11) تعریف دستگاه، ساخت مدل، معیار و بهینه‌ساز
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleAutoEncoder().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) حلقه آموزش
# ======================
num_epochs = 20  # به دلخواه تغییر دهید

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 13) ذخیره مدل (در صورت نیاز)
# ======================
torch.save(model.state_dict(), "autoencoder_cornea.pth")


In [5]:
# ======================
# 8) تعریف دیتالودر
# ======================
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 11) تعریف دستگاه، ساخت مدل، معیار و بهینه‌ساز
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleAutoEncoder().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) حلقه آموزش
# ======================
num_epochs = 20  # به دلخواه تغییر دهید

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 13) ذخیره مدل (در صورت نیاز)
# ======================
torch.save(model.state_dict(), "autoencoder_cornea.pth")
